In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving josaa_historical_clean.csv to josaa_historical_clean.csv


In [ ]:
df = pd.read_csv("josaa_historical_clean.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Rows: 29310
Columns: 129


,Institute,Academic Program Name,Quota,Seat Type,Gender,2016_R1_Opening,2016_R1_Closing,2016_R2_Opening,2016_R2_Closing,2016_R4_Opening,...,2025_R3_Opening,2025_R3_Closing,2025_R4_Opening,2025_R4_Closing,2025_R5_Opening,2025_R5_Closing,2025_R6_Opening,2025_R6_Closing,2026_R1_Opening,2026_R1_Closing
0,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,EWS,Gender-Neutral,NaN,NaN,NaN,NaN,NaN,...,12157.0,13158.0,12157.0,13283.0,12157.0,13458.0,14871.0,15343.0,NaN,NaN
1,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Female-only (including Supernumerary),NaN,NaN,NaN,NaN,NaN,...,25747.0,25747.0,25747.0,25747.0,25747.0,25747.0,31327.0,31327.0,NaN,NaN
2,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Gender-Neutral,10097.0,10493.0,11687.0,12605.0,12407.0,...,24512.0,28846.0,25173.0,29422.0,25173.0,29422.0,28525.0,31848.0,NaN,NaN
3,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Female-only (including Supernumerary),NaN,NaN,NaN,NaN,NaN,...,68243.0,68243.0,68243.0,68243.0,68243.0,68243.0,86651.0,86651.0,NaN,NaN
4,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Gender-Neutral,28101.0,32034.0,34691.0,39817.0,39515.0,...,60893.0,77136.0,60893.0,78837.0,60893.0,80704.0,60893.0,88476.0,NaN,NaN


In [ ]:
# ==========================================
# CREATE HISTORICAL TREND FEATURES
# ==========================================

# Only 2016-2025 Closing Rank columns
closing_columns = []

for column in df.columns:
    if "_Closing" in column:
        year = int(column[:4])

        if 2016 <= year <= 2025:
            closing_columns.append(column)

print("Closing rank columns:", len(closing_columns))


def calculate_trend_features(row):

    values = []
    years = []

    for column in closing_columns:

        value = row[column]

        if pd.notna(value):
            year = int(column[:4])

            values.append(float(value))
            years.append(year)

    # No historical data
    if len(values) == 0:
        return pd.Series({
            "Historical_Data_Points": 0,
            "Average_Closing_Rank": np.nan,
            "Recent_Average_Closing": np.nan,
            "Minimum_Closing_Rank": np.nan,
            "Maximum_Closing_Rank": np.nan,
            "Closing_Rank_Volatility": np.nan,
            "Trend_Slope": np.nan,
            "Trend_Direction": "No Data"
        })

    values = np.array(values)
    years = np.array(years)

    # Average closing rank
    average = np.mean(values)

    # Recent average: 2023-2025
    recent_values = values[years >= 2023]

    if len(recent_values) > 0:
        recent_average = np.mean(recent_values)
    else:
        recent_average = np.nan

    # Minimum and maximum
    minimum = np.min(values)
    maximum = np.max(values)

    # Volatility
    volatility = np.std(values)

    # Safe trend calculation
    if len(values) >= 2 and len(np.unique(years)) >= 2:

        year_mean = np.mean(years)
        value_mean = np.mean(values)

        numerator = np.sum(
            (years - year_mean) * (values - value_mean)
        )

        denominator = np.sum(
            (years - year_mean) ** 2
        )

        if denominator != 0:
            slope = numerator / denominator
        else:
            slope = 0

    else:
        slope = 0

    # Trend direction
    if slope > 10:
        direction = "Rank Increasing"

    elif slope < -10:
        direction = "Rank Decreasing"

    else:
        direction = "Stable"

    return pd.Series({
        "Historical_Data_Points": len(values),
        "Average_Closing_Rank": average,
        "Recent_Average_Closing": recent_average,
        "Minimum_Closing_Rank": minimum,
        "Maximum_Closing_Rank": maximum,
        "Closing_Rank_Volatility": volatility,
        "Trend_Slope": slope,
        "Trend_Direction": direction
    })


print("Creating trend features...")

trend_features = df.apply(
    calculate_trend_features,
    axis=1
)

# Add features to dataset
df_ml = pd.concat(
    [df, trend_features],
    axis=1
)

print("\nTrend features created successfully!")

print("Original columns:", df.shape[1])
print("New columns:", df_ml.shape[1])

display(
    df_ml[
        [
            "Institute",
            "Academic Program Name",
            "Average_Closing_Rank",
            "Recent_Average_Closing",
            "Minimum_Closing_Rank",
            "Maximum_Closing_Rank",
            "Closing_Rank_Volatility",
            "Trend_Slope",
            "Trend_Direction",
            "Historical_Data_Points"
        ]
    ].head()
)

Closing rank columns: 61
Creating trend features...

Trend features created successfully!
Original columns: 129
New columns: 137


,Institute,Academic Program Name,Average_Closing_Rank,Recent_Average_Closing,Minimum_Closing_Rank,Maximum_Closing_Rank,Closing_Rank_Volatility,Trend_Slope,Trend_Direction,Historical_Data_Points
0,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",10985.047619,12881.529412,5938.0,15343.0,2456.193435,968.820329,Rank Increasing,42
1,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",21939.142857,25148.294118,15996.0,31327.0,3620.327311,1263.419540,Rank Increasing,49
2,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",19759.836066,25906.823529,10135.0,31848.0,5502.724685,1782.051162,Rank Increasing,61
3,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",60972.673469,67886.705882,32674.0,86651.0,9584.819870,3236.615253,Rank Increasing,49
4,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",59492.016393,73191.117647,30754.0,88476.0,14174.070058,4331.736921,Rank Increasing,61


In [ ]:
# ==========================================
# FEATURE QUALITY CHECK
# ==========================================

feature_columns = [
    "Historical_Data_Points",
    "Average_Closing_Rank",
    "Recent_Average_Closing",
    "Minimum_Closing_Rank",
    "Maximum_Closing_Rank",
    "Closing_Rank_Volatility",
    "Trend_Slope",
    "Trend_Direction"
]

print("====================================")
print("FEATURE QUALITY CHECK")
print("====================================")

print("\nMissing values:")
print(df_ml[feature_columns].isna().sum())

print("\nTrend Direction:")
print(df_ml["Trend_Direction"].value_counts(dropna=False))

print("\nHistorical Data Points:")
print(df_ml["Historical_Data_Points"].describe())

print("\nNumerical Feature Summary:")
display(
    df_ml[
        [
            "Average_Closing_Rank",
            "Recent_Average_Closing",
            "Minimum_Closing_Rank",
            "Maximum_Closing_Rank",
            "Closing_Rank_Volatility",
            "Trend_Slope"
        ]
    ].describe()
)


FEATURE QUALITY CHECK

Missing values:
Historical_Data_Points         0
Average_Closing_Rank       13085
Recent_Average_Closing     15926
Minimum_Closing_Rank       13085
Maximum_Closing_Rank       13085
Closing_Rank_Volatility    13085
Trend_Slope                13085
Trend_Direction                0
dtype: int64

Trend Direction:
Trend_Direction
No Data            13085
Rank Increasing     9407
Stable              3636
Rank Decreasing     3182
Name: count, dtype: int64

Historical Data Points:
count    29310.000000
mean        17.415763
std         21.695407
min          0.000000
25%          0.000000
50%          6.000000
75%         35.000000
max         61.000000
Name: Historical_Data_Points, dtype: float64

Numerical Feature Summary:


,Average_Closing_Rank,Recent_Average_Closing,Minimum_Closing_Rank,Maximum_Closing_Rank,Closing_Rank_Volatility,Trend_Slope
count,1.622500e+04,1.338400e+04,16225.000000,1.622500e+04,16225.000000,16225.000000
mean,1.327587e+04,1.461202e+04,8879.949131,1.884657e+04,2752.259134,258.613932
std,4.129283e+04,4.260907e+04,26425.814245,6.382005e+04,12433.828115,8382.030866
min,1.000000e+00,1.000000e+00,0.010000,1.000000e+00,0.000000,-638033.000000
25%,1.154265e+03,1.513794e+03,708.000000,1.553000e+03,166.000000,0.000000
50%,3.936182e+03,4.670342e+03,2802.000000,4.962000e+03,548.173523,51.393258
75%,1.121274e+04,1.308994e+04,7977.000000,1.445800e+04,1574.315351,356.834964
max,1.106307e+06,1.106307e+06,937704.000000,1.368129e+06,341045.414942,276004.142857


In [ ]:
# ==========================================
# TREND FEATURES V2
# YEAR-WISE HISTORICAL TREND
# ==========================================

# 2016-2025 Closing Rank columns
closing_columns = []

for column in df.columns:

    if "_Closing" in column:

        year = int(column[:4])

        if 2016 <= year <= 2025:
            closing_columns.append(column)


# Extract available years
years_available = sorted(
    list(set(int(col[:4]) for col in closing_columns))
)

print("Years used:", years_available)


def calculate_trend_v2(row):

    yearly_values = {}

    # --------------------------------------
    # Step 1: Calculate one value per year
    # --------------------------------------

    for year in years_available:

        year_columns = [
            col for col in closing_columns
            if col.startswith(str(year) + "_")
        ]

        values = []

        for col in year_columns:

            value = row[col]

            if pd.notna(value):
                values.append(float(value))

        if len(values) > 0:

            # Average closing rank of all rounds in that year
            yearly_values[year] = np.mean(values)


    # --------------------------------------
    # No historical data
    # --------------------------------------

    if len(yearly_values) == 0:

        return pd.Series({
            "Historical_Years": 0,
            "Average_Closing_Rank_V2": np.nan,
            "Recent_Average_Closing_V2": np.nan,
            "Minimum_Closing_Rank_V2": np.nan,
            "Maximum_Closing_Rank_V2": np.nan,
            "Closing_Rank_Volatility_V2": np.nan,
            "Trend_Slope_V2": np.nan,
            "Trend_Direction_V2": "No Data"
        })


    # --------------------------------------
    # Convert to arrays
    # --------------------------------------

    years = np.array(list(yearly_values.keys()))

    values = np.array(list(yearly_values.values()))


    # --------------------------------------
    # Basic statistics
    # --------------------------------------

    average = np.mean(values)

    minimum = np.min(values)

    maximum = np.max(values)

    volatility = np.std(values)


    # --------------------------------------
    # Recent average: 2023-2025
    # --------------------------------------

    recent_values = [
        value
        for year, value in yearly_values.items()
        if year >= 2023
    ]

    if len(recent_values) > 0:
        recent_average = np.mean(recent_values)
    else:
        recent_average = np.nan


    # --------------------------------------
    # Trend slope
    # --------------------------------------

    if len(values) >= 2:

        year_mean = np.mean(years)

        value_mean = np.mean(values)

        numerator = np.sum(
            (years - year_mean) *
            (values - value_mean)
        )

        denominator = np.sum(
            (years - year_mean) ** 2
        )

        if denominator != 0:
            slope = numerator / denominator
        else:
            slope = 0

    else:

        slope = 0


    # --------------------------------------
    # Trend direction
    # --------------------------------------

    if slope > 100:

        direction = "Rank Increasing"

    elif slope < -100:

        direction = "Rank Decreasing"

    else:

        direction = "Stable"


    return pd.Series({

        "Historical_Years": len(yearly_values),

        "Average_Closing_Rank_V2": average,

        "Recent_Average_Closing_V2": recent_average,

        "Minimum_Closing_Rank_V2": minimum,

        "Maximum_Closing_Rank_V2": maximum,

        "Closing_Rank_Volatility_V2": volatility,

        "Trend_Slope_V2": slope,

        "Trend_Direction_V2": direction
    })


print("Creating improved trend features...")

trend_features_v2 = df.apply(
    calculate_trend_v2,
    axis=1
)


# Combine with original dataset
df_ml_v2 = pd.concat(
    [df, trend_features_v2],
    axis=1
)


print("\n====================================")
print("TREND FEATURES V2 CREATED")
print("====================================")

print("Rows:", df_ml_v2.shape[0])
print("Columns:", df_ml_v2.shape[1])


display(
    df_ml_v2[
        [
            "Institute",
            "Academic Program Name",
            "Historical_Years",
            "Average_Closing_Rank_V2",
            "Recent_Average_Closing_V2",
            "Minimum_Closing_Rank_V2",
            "Maximum_Closing_Rank_V2",
            "Closing_Rank_Volatility_V2",
            "Trend_Slope_V2",
            "Trend_Direction_V2"
        ]
    ].head(10)
)

Years used: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Creating improved trend features...

TREND FEATURES V2 CREATED
Rows: 29310
Columns: 137


,Institute,Academic Program Name,Historical_Years,Average_Closing_Rank_V2,Recent_Average_Closing_V2,Minimum_Closing_Rank_V2,Maximum_Closing_Rank_V2,Closing_Rank_Volatility_V2,Trend_Slope_V2,Trend_Direction_V2
0,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",7,11132.311565,12886.155556,6779.714286,12981.666667,2075.418198,942.694898,Rank Increasing
1,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",8,22158.551786,25146.344444,18061.857143,26531.000000,2940.731071,1260.554138,Rank Increasing
2,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",10,19998.106190,25890.322222,12649.571429,28297.333333,5118.411412,1751.989928,Rank Increasing
3,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",8,61562.992857,67898.544444,49803.000000,71311.000000,7846.761553,3190.995465,Rank Increasing
4,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",10,60029.984762,73047.222222,40354.714286,77623.000000,12703.137349,4240.622742,Rank Increasing
5,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",8,1954.833333,2069.333333,1023.000000,3118.000000,656.710680,-4.414815,Stable
6,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",4,11355.636905,NaN,9710.714286,12525.000000,1023.035465,661.402381,Rank Increasing
7,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",10,10398.107143,12136.622222,7701.714286,13655.166667,1753.630494,576.804242,Rank Increasing
8,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",10,4371.872857,5711.155556,2696.000000,5911.500000,1079.221022,266.540866,Rank Increasing
9,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",1,24787.333333,NaN,24787.333333,24787.333333,0.000000,0.000000,Stable


In [ ]:
# ==========================================
# TREND FEATURES V2 - QUALITY CHECK
# ==========================================

v2_features = [
    "Historical_Years",
    "Average_Closing_Rank_V2",
    "Recent_Average_Closing_V2",
    "Minimum_Closing_Rank_V2",
    "Maximum_Closing_Rank_V2",
    "Closing_Rank_Volatility_V2",
    "Trend_Slope_V2",
    "Trend_Direction_V2"
]

print("====================================")
print("TREND FEATURES V2 QUALITY CHECK")
print("====================================")


# --------------------------------------
# Missing values
# --------------------------------------

print("\nMissing values:")

print(
    df_ml_v2[v2_features].isna().sum()
)


# --------------------------------------
# Trend direction
# --------------------------------------

print("\nTrend Direction:")

print(
    df_ml_v2["Trend_Direction_V2"]
    .value_counts(dropna=False)
)


# --------------------------------------
# Historical years
# --------------------------------------

print("\nHistorical Years:")

print(
    df_ml_v2["Historical_Years"].describe()
)


# --------------------------------------
# Numerical feature summary
# --------------------------------------

print("\nNumerical Feature Summary:")

display(
    df_ml_v2[
        [
            "Average_Closing_Rank_V2",
            "Recent_Average_Closing_V2",
            "Minimum_Closing_Rank_V2",
            "Maximum_Closing_Rank_V2",
            "Closing_Rank_Volatility_V2",
            "Trend_Slope_V2"
        ]
    ].describe()
)

TREND FEATURES V2 QUALITY CHECK

Missing values:
Historical_Years                  0
Average_Closing_Rank_V2       13085
Recent_Average_Closing_V2     15926
Minimum_Closing_Rank_V2       13085
Maximum_Closing_Rank_V2       13085
Closing_Rank_Volatility_V2    13085
Trend_Slope_V2                13085
Trend_Direction_V2                0
dtype: int64

Trend Direction:
Trend_Direction_V2
No Data            13085
Stable              7398
Rank Increasing     6928
Rank Decreasing     1899
Name: count, dtype: int64

Historical Years:
count    29310.000000
mean         2.963562
std          3.578433
min          0.000000
25%          0.000000
50%          1.000000
75%          6.000000
max         10.000000
Name: Historical_Years, dtype: float64

Numerical Feature Summary:


,Average_Closing_Rank_V2,Recent_Average_Closing_V2,Minimum_Closing_Rank_V2,Maximum_Closing_Rank_V2,Closing_Rank_Volatility_V2,Trend_Slope_V2
count,1.622500e+04,1.338400e+04,1.622500e+04,1.622500e+04,16225.000000,16225.000000
mean,1.333855e+04,1.465163e+04,1.033160e+04,1.669990e+04,2195.250024,278.272814
std,4.160084e+04,4.298010e+04,3.109378e+04,5.586658e+04,10791.960072,8592.000722
min,1.000000e+00,1.000000e+00,1.000000e-02,1.000000e+00,0.000000,-638033.000000
25%,1.161811e+03,1.510000e+03,8.000000e+02,1.457833e+03,66.500000,0.000000
50%,3.951386e+03,4.671939e+03,3.103000e+03,4.616000e+03,373.214865,51.838776
75%,1.126000e+04,1.308793e+04,9.047143e+03,1.325167e+04,1153.748028,355.433333
max,1.106307e+06,1.106307e+06,1.106307e+06,1.235927e+06,319016.500000,308966.750000


In [ ]:
# ==========================================
# CHECK EXTREME RANK VALUES
# ==========================================

print("====================================")
print("EXTREME RANK VALUE CHECK")
print("====================================")


# --------------------------------------
# Highest average closing ranks
# --------------------------------------

print("\nTop 20 highest Average Closing Rank:")

display(
    df_ml_v2[
        [
            "Institute",
            "Academic Program Name",
            "Quota",
            "Seat Type",
            "Gender",
            "Historical_Years",
            "Average_Closing_Rank_V2",
            "Maximum_Closing_Rank_V2",
            "Trend_Slope_V2"
        ]
    ]
    .sort_values(
        "Average_Closing_Rank_V2",
        ascending=False
    )
    .head(20)
)


# --------------------------------------
# Lowest trend slopes
# --------------------------------------

print("\n20 most negative Trend Slopes:")

display(
    df_ml_v2[
        [
            "Institute",
            "Academic Program Name",
            "Quota",
            "Seat Type",
            "Gender",
            "Historical_Years",
            "Average_Closing_Rank_V2",
            "Trend_Slope_V2"
        ]
    ]
    .sort_values(
        "Trend_Slope_V2",
        ascending=True
    )
    .head(20)
)

EXTREME RANK VALUE CHECK

Top 20 highest Average Closing Rank:


,Institute,Academic Program Name,Quota,Seat Type,Gender,Historical_Years,Average_Closing_Rank_V2,Maximum_Closing_Rank_V2,Trend_Slope_V2
23170,"National Institute of Technology, Mizoram","Chemical Science and Technology (4 Years, Bach...",HS,OPEN,Gender-Neutral,1,1.106307e+06,1.106307e+06,0.000000
23338,"National Institute of Technology, Mizoram","Mechanical Engineering (4 Years, Bachelor of T...",HS,OPEN,Gender-Neutral,10,9.111634e+05,1.235927e+06,9645.346032
23322,"National Institute of Technology, Mizoram","Mathematics and Computing (4 Years, Bachelor o...",HS,OPEN,Female-only (including Supernumerary),3,8.830165e+05,1.115070e+06,308966.750000
23275,"National Institute of Technology, Mizoram",Electrical and Electronics Engineering (4 Year...,HS,OPEN,Gender-Neutral,7,8.733828e+05,9.881620e+05,-25145.959184
23274,"National Institute of Technology, Mizoram",Electrical and Electronics Engineering (4 Year...,HS,OPEN,Female-only (including Supernumerary),4,8.587653e+05,1.029730e+06,-35613.333333
23250,"National Institute of Technology, Mizoram","Electrical Engineering (4 Years, Bachelor of T...",HS,OPEN,Gender-Neutral,3,8.372186e+05,1.051316e+06,52581.083333
23294,"National Institute of Technology, Mizoram",Electronics and Communication Engineering (4 Y...,HS,OPEN,Gender-Neutral,10,8.297070e+05,1.046638e+06,24286.790563
21208,National Institute of Technology Sikkim,"Civil Engineering (4 Years, Bachelor of Techno...",HS,OPEN,Gender-Neutral,10,8.180077e+05,1.100355e+06,-17013.096566
23249,"National Institute of Technology, Mizoram","Electrical Engineering (4 Years, Bachelor of T...",HS,OPEN,Female-only (including Supernumerary),3,8.127801e+05,1.221717e+06,284590.916667
23169,"National Institute of Technology, Mizoram","Chemical Science and Technology (4 Years, Bach...",HS,OPEN,Female-only (including Supernumerary),1,8.001380e+05,8.001380e+05,0.000000



20 most negative Trend Slopes:


,Institute,Academic Program Name,Quota,Seat Type,Gender,Historical_Years,Average_Closing_Rank_V2,Trend_Slope_V2
21176,National Institute of Technology Sikkim,Artificial Intelligence and Machine Learning (...,HS,OPEN,Female-only (including Supernumerary),2,529596.500000,-638033.000000
17966,National Institute of Technology Goa,"Civil Engineering (4 Years, Bachelor of Techno...",HS,OPEN,Female-only (including Supernumerary),2,421241.000000,-506116.000000
21200,National Institute of Technology Sikkim,"Bio Technology (4 Years, Bachelor of Technology)",HS,OPEN,Gender-Neutral,2,691370.600000,-311899.200000
24650,"National Institute of Technology, Srinagar","Chemical Engineering (4 Years, Bachelor of Tec...",HS,OBC-NCL,Female-only (including Supernumerary),2,308779.250000,-133969.500000
25095,"National Institute of Technology, Srinagar",Metallurgical and Materials Engineering (4 Yea...,HS,OPEN,Female-only (including Supernumerary),2,204910.142857,-104016.571429
24660,"National Institute of Technology, Srinagar","Chemical Engineering (4 Years, Bachelor of Tec...",JK,OBC-NCL,Female-only (including Supernumerary),2,163977.500000,-98692.500000
24652,"National Institute of Technology, Srinagar","Chemical Engineering (4 Years, Bachelor of Tec...",HS,OPEN,Female-only (including Supernumerary),2,171430.071429,-69838.142857
21271,National Institute of Technology Sikkim,Electrical and Electronics Engineering (4 Year...,HS,OPEN,Gender-Neutral,10,669795.538571,-66894.346522
24968,"National Institute of Technology, Srinagar","Information Technology (4 Years, Bachelor of T...",HS,OPEN,Female-only (including Supernumerary),2,98369.357143,-64232.714286
21301,National Institute of Technology Sikkim,Electronics and Communication Engineering (4 Y...,HS,OPEN,Gender-Neutral,10,499104.908571,-63461.574661


In [ ]:
# ==========================================
# STEP 1: CREATE TIME-BASED ML TARGET DATA
# ==========================================

print("====================================")
print("CREATING TIME-BASED ML TARGET DATA")
print("====================================")

# We will predict Round-1 Closing Rank
target_years = list(range(2019, 2027))

target_columns = []

for year in target_years:

    column = f"{year}_R1_Closing"

    if column in df.columns:
        target_columns.append(column)


print("\nTarget columns found:")
print(target_columns)


# --------------------------------------
# Check how many values each target has
# --------------------------------------

print("\nAvailable target values:")

for column in target_columns:

    count = df[column].notna().sum()

    print(
        column,
        "->",
        count,
        "records"
    )


# --------------------------------------
# Create basic ML dataset
# --------------------------------------

ml_dataset = df[
    [
        "Institute",
        "Academic Program Name",
        "Quota",
        "Seat Type",
        "Gender"
    ] + target_columns
].copy()


print("\n====================================")
print("ML DATASET CREATED")
print("====================================")

print("Rows:", ml_dataset.shape[0])
print("Columns:", ml_dataset.shape[1])

display(ml_dataset.head())

CREATING TIME-BASED ML TARGET DATA

Target columns found:
['2019_R1_Closing', '2020_R1_Closing', '2021_R1_Closing', '2022_R1_Closing', '2023_R1_Closing', '2024_R1_Closing', '2025_R1_Closing', '2026_R1_Closing']

Available target values:
2019_R1_Closing -> 8524 records
2020_R1_Closing -> 9013 records
2021_R1_Closing -> 9097 records
2022_R1_Closing -> 9929 records
2023_R1_Closing -> 10695 records
2024_R1_Closing -> 11484 records
2025_R1_Closing -> 12131 records
2026_R1_Closing -> 13085 records

ML DATASET CREATED
Rows: 29310
Columns: 13


,Institute,Academic Program Name,Quota,Seat Type,Gender,2019_R1_Closing,2020_R1_Closing,2021_R1_Closing,2022_R1_Closing,2023_R1_Closing,2024_R1_Closing,2025_R1_Closing,2026_R1_Closing
0,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,EWS,Gender-Neutral,5938.0,8363.0,9019.0,9084.0,10211.0,11605.0,10958.0,NaN
1,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Female-only (including Supernumerary),16063.0,17172.0,16628.0,18813.0,19132.0,23136.0,24871.0,NaN
2,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Gender-Neutral,14939.0,14653.0,18345.0,19461.0,20253.0,21415.0,23389.0,NaN
3,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Female-only (including Supernumerary),44720.0,48816.0,53080.0,62283.0,57456.0,66674.0,68243.0,NaN
4,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Gender-Neutral,41452.0,48265.0,51729.0,58716.0,58067.0,64102.0,66923.0,NaN


In [ ]:
# ==========================================
# STEP 2: CREATE ROLLING HISTORICAL FEATURES
# ==========================================

print("====================================")
print("CREATING ROLLING HISTORICAL FEATURES")
print("====================================")


# Years that can be used as prediction targets
prediction_years = list(range(2019, 2027))


def create_historical_features(row, target_year):

    # --------------------------------------
    # Only use years BEFORE target year
    # --------------------------------------

    previous_years = list(
        range(2016, target_year)
    )

    yearly_values = []

    for year in previous_years:

        column = f"{year}_R1_Closing"

        if column in df.columns:

            value = row[column]

            if pd.notna(value):
                yearly_values.append(float(value))


    # --------------------------------------
    # No previous historical data
    # --------------------------------------

    if len(yearly_values) == 0:

        return {
            "Historical_Count": 0,
            "Historical_Mean": np.nan,
            "Historical_Median": np.nan,
            "Historical_Min": np.nan,
            "Historical_Max": np.nan,
            "Historical_Std": np.nan,
            "Recent_Mean": np.nan,
            "Trend_Slope": np.nan
        }


    values = np.array(yearly_values)


    # --------------------------------------
    # Basic statistics
    # --------------------------------------

    historical_mean = np.mean(values)

    historical_median = np.median(values)

    historical_min = np.min(values)

    historical_max = np.max(values)

    historical_std = np.std(values)


    # --------------------------------------
    # Recent historical average
    # --------------------------------------

    recent_years = list(
        range(max(2016, target_year - 3), target_year)
    )

    recent_values = []

    for year in recent_years:

        column = f"{year}_R1_Closing"

        if column in df.columns:

            value = row[column]

            if pd.notna(value):
                recent_values.append(float(value))


    if len(recent_values) > 0:
        recent_mean = np.mean(recent_values)
    else:
        recent_mean = np.nan


    # --------------------------------------
    # Trend slope
    # --------------------------------------

    years_used = []

    values_used = []

    for year in previous_years:

        column = f"{year}_R1_Closing"

        if column in df.columns:

            value = row[column]

            if pd.notna(value):

                years_used.append(year)
                values_used.append(float(value))


    if len(values_used) >= 2:

        years_array = np.array(years_used)

        values_array = np.array(values_used)

        year_mean = np.mean(years_array)

        value_mean = np.mean(values_array)

        numerator = np.sum(
            (years_array - year_mean) *
            (values_array - value_mean)
        )

        denominator = np.sum(
            (years_array - year_mean) ** 2
        )

        if denominator != 0:
            trend_slope = numerator / denominator
        else:
            trend_slope = 0

    else:

        trend_slope = 0


    return {
        "Historical_Count": len(values),
        "Historical_Mean": historical_mean,
        "Historical_Median": historical_median,
        "Historical_Min": historical_min,
        "Historical_Max": historical_max,
        "Historical_Std": historical_std,
        "Recent_Mean": recent_mean,
        "Trend_Slope": trend_slope
    }


# ==========================================
# CREATE TRAINING RECORDS
# ==========================================

training_records = []


for target_year in prediction_years:

    print(
        "Creating records for:",
        target_year
    )


    target_column = f"{target_year}_R1_Closing"


    for index, row in df.iterrows():

        target = row[target_column]


        # We need an actual target value
        if pd.isna(target):
            continue


        features = create_historical_features(
            row,
            target_year
        )


        record = {

            "Target_Year": target_year,

            "Institute": row["Institute"],

            "Academic Program Name":
                row["Academic Program Name"],

            "Quota":
                row["Quota"],

            "Seat Type":
                row["Seat Type"],

            "Gender":
                row["Gender"],

            "Target_Closing_Rank":
                float(target)
        }


        record.update(features)


        training_records.append(record)


# ==========================================
# CREATE FINAL ML DATAFRAME
# ==========================================

ml_time_dataset = pd.DataFrame(
    training_records
)


print("\n====================================")
print("TIME-BASED ML DATASET CREATED")
print("====================================")

print(
    "Rows:",
    ml_time_dataset.shape[0]
)

print(
    "Columns:",
    ml_time_dataset.shape[1]
)


print("\nRecords by target year:")

print(
    ml_time_dataset["Target_Year"]
    .value_counts()
    .sort_index()
)


display(
    ml_time_dataset.head(10)
)

CREATING ROLLING HISTORICAL FEATURES
Creating records for: 2019
Creating records for: 2020
Creating records for: 2021
Creating records for: 2022
Creating records for: 2023
Creating records for: 2024
Creating records for: 2025
Creating records for: 2026

TIME-BASED ML DATASET CREATED
Rows: 83958
Columns: 15

Records by target year:
Target_Year
2019     8524
2020     9013
2021     9097
2022     9929
2023    10695
2024    11484
2025    12131
2026    13085
Name: count, dtype: int64


,Target_Year,Institute,Academic Program Name,Quota,Seat Type,Gender,Target_Closing_Rank,Historical_Count,Historical_Mean,Historical_Median,Historical_Min,Historical_Max,Historical_Std,Recent_Mean,Trend_Slope
0,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,EWS,Gender-Neutral,5938.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Female-only (including Supernumerary),16063.0,1,15996.000000,15996.0,15996.0,15996.0,0.000000,15996.000000,0.0
2,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Gender-Neutral,14939.0,3,10593.000000,10493.0,10135.0,11151.0,420.764384,10593.000000,329.0
3,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Female-only (including Supernumerary),44720.0,1,32674.000000,32674.0,32674.0,32674.0,0.000000,32674.000000,0.0
4,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Gender-Neutral,41452.0,3,34418.666667,32034.0,30754.0,40468.0,4309.325186,34418.666667,4217.0
5,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN (PwD),Gender-Neutral,1259.0,2,1192.000000,1192.0,1120.0,1264.0,72.000000,1192.000000,72.0
6,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,SC,Female-only (including Supernumerary),8825.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,SC,Gender-Neutral,7063.0,3,6471.000000,6364.0,5927.0,7122.0,493.688836,6471.000000,379.0
8,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,ST,Gender-Neutral,3751.0,3,2682.333333,2975.0,1985.0,3087.0,495.204559,2682.333333,551.0
9,2019,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",HS,EWS,Gender-Neutral,21530.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ==========================================
# STEP 3: TIME-BASED TRAIN / VALIDATION SPLIT
# ==========================================

print("====================================")
print("TIME-BASED DATA SPLIT")
print("====================================")


# Training data: 2019-2024
train_data = ml_time_dataset[
    ml_time_dataset["Target_Year"] <= 2024
].copy()


# Validation data: 2025
validation_data = ml_time_dataset[
    ml_time_dataset["Target_Year"] == 2025
].copy()


# Final prediction data: 2026
test_2026 = ml_time_dataset[
    ml_time_dataset["Target_Year"] == 2026
].copy()


print("\nTraining records:", len(train_data))

print(
    "Validation records:",
    len(validation_data)
)

print(
    "2026 prediction records:",
    len(test_2026)
)


print("\nTraining years:")

print(
    sorted(train_data["Target_Year"].unique())
)


print("\nValidation year:")

print(
    sorted(validation_data["Target_Year"].unique())
)


print("\nFinal prediction year:")

print(
    sorted(test_2026["Target_Year"].unique())
)

TIME-BASED DATA SPLIT

Training records: 58742
Validation records: 12131
2026 prediction records: 13085

Training years:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Validation year:
[np.int64(2025)]

Final prediction year:
[np.int64(2026)]


In [ ]:
# ==========================================
# STEP 4: PREPARE X AND Y
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


print("====================================")
print("PREPARING FEATURES AND TARGET")
print("====================================")


# --------------------------------------
# Feature columns
# --------------------------------------

categorical_features = [
    "Institute",
    "Academic Program Name",
    "Quota",
    "Seat Type",
    "Gender"
]


numerical_features = [
    "Historical_Count",
    "Historical_Mean",
    "Historical_Median",
    "Historical_Min",
    "Historical_Max",
    "Historical_Std",
    "Recent_Mean",
    "Trend_Slope"
]


all_features = (
    categorical_features +
    numerical_features
)


target_column = "Target_Closing_Rank"


# --------------------------------------
# Create X and y
# --------------------------------------

X_train = train_data[all_features]

y_train = train_data[target_column]


X_validation = validation_data[all_features]

y_validation = validation_data[target_column]


X_2026 = test_2026[all_features]

y_2026 = test_2026[target_column]


# --------------------------------------
# Preprocessing
# --------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nTraining X shape:")
print(X_train.shape)

print("\nTraining y shape:")
print(y_train.shape)

print("\nValidation X shape:")
print(X_validation.shape)

print("\n2026 X shape:")
print(X_2026.shape)

print("\nTarget statistics:")
print(y_train.describe())

PREPARING FEATURES AND TARGET
Categorical features:
['Institute', 'Academic Program Name', 'Quota', 'Seat Type', 'Gender']

Numerical features:
['Historical_Count', 'Historical_Mean', 'Historical_Median', 'Historical_Min', 'Historical_Max', 'Historical_Std', 'Recent_Mean', 'Trend_Slope']

Training X shape:
(58742, 13)

Training y shape:
(58742,)

Validation X shape:
(12131, 13)

2026 X shape:
(13085, 13)

Target statistics:
count    5.874200e+04
mean     1.213552e+04
std      3.556657e+04
min      1.000000e+00
25%      1.339000e+03
50%      4.093000e+03
75%      1.088550e+04
max      1.070084e+06
Name: Target_Closing_Rank, dtype: float64


In [ ]:
# ==========================================
# STEP 5: RANDOM FOREST MODEL
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("====================================")
print("TRAINING RANDOM FOREST")
print("====================================")

# Create model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Complete ML pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

# Train
print("Training model...")

rf_pipeline.fit(
    X_train,
    y_train
)

print("Model training completed!")


# --------------------------------------
# Validation prediction: 2025
# --------------------------------------

print("\nMaking 2025 predictions...")

y_pred_2025 = rf_pipeline.predict(
    X_validation
)


# --------------------------------------
# Evaluation
# --------------------------------------

mae_2025 = mean_absolute_error(
    y_validation,
    y_pred_2025
)

rmse_2025 = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_2025
    )
)

r2_2025 = r2_score(
    y_validation,
    y_pred_2025
)


print("\n====================================")
print("2025 VALIDATION RESULTS")
print("====================================")

print("MAE :", round(mae_2025, 2))
print("RMSE:", round(rmse_2025, 2))
print("R²  :", round(r2_2025, 4))

TRAINING RANDOM FOREST
Training model...
Model training completed!

Making 2025 predictions...

2025 VALIDATION RESULTS
MAE : 2706.86
RMSE: 16115.09
R²  : 0.8275


In [ ]:
# ==========================================
# STEP 6: RANDOM FOREST WITH LOG TARGET
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("====================================")
print("RANDOM FOREST + LOG TARGET")
print("====================================")


# --------------------------------------
# Transform target
# --------------------------------------

y_train_log = np.log1p(y_train)


print("Original target example:")
print(y_train.head().values)

print("\nLog-transformed example:")
print(y_train_log.head().values)


# --------------------------------------
# Create model
# --------------------------------------

rf_log_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_log_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_log_model)
    ]
)


# --------------------------------------
# Train
# --------------------------------------

print("\nTraining log-target model...")

rf_log_pipeline.fit(
    X_train,
    y_train_log
)

print("Model training completed!")


# --------------------------------------
# Predict 2025
# --------------------------------------

print("\nMaking 2025 predictions...")

y_pred_2025_log = rf_log_pipeline.predict(
    X_validation
)


# Convert back to original rank
y_pred_2025_log_original = np.expm1(
    y_pred_2025_log
)


# --------------------------------------
# Evaluation
# --------------------------------------

mae_2025_log = mean_absolute_error(
    y_validation,
    y_pred_2025_log_original
)

rmse_2025_log = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_2025_log_original
    )
)

r2_2025_log = r2_score(
    y_validation,
    y_pred_2025_log_original
)


print("\n====================================")
print("2025 LOG-TARGET RESULTS")
print("====================================")

print("MAE :", round(mae_2025_log, 2))
print("RMSE:", round(rmse_2025_log, 2))
print("R²  :", round(r2_2025_log, 4))

RANDOM FOREST + LOG TARGET
Original target example:
[ 5938. 16063. 14939. 44720. 41452.]

Log-transformed example:
[ 8.68929605  9.68433602  9.61179746 10.70819847 10.63231553]

Training log-target model...
Model training completed!

Making 2025 predictions...

2025 LOG-TARGET RESULTS
MAE : 2694.34
RMSE: 16183.35
R²  : 0.826


In [ ]:
# ==========================================
# STEP 7: ADD TARGET YEAR
# ==========================================

print("====================================")
print("ADDING TARGET YEAR FEATURE")
print("====================================")


# --------------------------------------
# Add Target_Year to numerical features
# --------------------------------------

numerical_features_v2 = [
    "Target_Year",
    "Historical_Count",
    "Historical_Mean",
    "Historical_Median",
    "Historical_Min",
    "Historical_Max",
    "Historical_Std",
    "Recent_Mean",
    "Trend_Slope"
]


all_features_v2 = (
    categorical_features +
    numerical_features_v2
)


# --------------------------------------
# Create new X datasets
# --------------------------------------

X_train_v2 = train_data[all_features_v2]

X_validation_v2 = validation_data[all_features_v2]

X_2026_v2 = test_2026[all_features_v2]


# --------------------------------------
# New preprocessor
# --------------------------------------

numeric_transformer_v2 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer_v2,
            numerical_features_v2
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# --------------------------------------
# Random Forest
# --------------------------------------

rf_model_v2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_pipeline_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", rf_model_v2)
    ]
)


# --------------------------------------
# Train
# --------------------------------------

print("Training model...")

rf_pipeline_v2.fit(
    X_train_v2,
    y_train
)

print("Model training completed!")


# --------------------------------------
# Predict 2025
# --------------------------------------

print("\nMaking 2025 predictions...")

y_pred_2025_v2 = rf_pipeline_v2.predict(
    X_validation_v2
)


# --------------------------------------
# Evaluation
# --------------------------------------

mae_2025_v2 = mean_absolute_error(
    y_validation,
    y_pred_2025_v2
)

rmse_2025_v2 = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_2025_v2
    )
)

r2_2025_v2 = r2_score(
    y_validation,
    y_pred_2025_v2
)


print("\n====================================")
print("2025 RESULTS - MODEL WITH YEAR")
print("====================================")

print("MAE :", round(mae_2025_v2, 2))
print("RMSE:", round(rmse_2025_v2, 2))
print("R²  :", round(r2_2025_v2, 4))

ADDING TARGET YEAR FEATURE
Training model...
Model training completed!

Making 2025 predictions...

2025 RESULTS - MODEL WITH YEAR
MAE : 2680.03
RMSE: 16084.77
R²  : 0.8281


In [ ]:
# ==========================================
# STEP 8: FEATURE IMPORTANCE
# ==========================================

print("====================================")
print("FEATURE IMPORTANCE")
print("====================================")


# Get the trained Random Forest
model = rf_pipeline_v2.named_steps["model"]

# Get preprocessor
processor = rf_pipeline_v2.named_steps["preprocessor"]


# Get feature names after preprocessing
feature_names = processor.get_feature_names_out()


# Get importance values
importances = model.feature_importances_


# Create DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})


# Sort from highest to lowest
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)


print("\nTop 20 Most Important Features:")
display(
    importance_df.head(20)
)

FEATURE IMPORTANCE

Top 20 Most Important Features:


,Feature,Importance
2,num__Historical_Mean,0.371351
7,num__Recent_Mean,0.248290
5,num__Historical_Max,0.162487
3,num__Historical_Median,0.078099
8,num__Trend_Slope,0.013904
4,num__Historical_Min,0.013059
427,cat__Quota_LA,0.013051
0,num__Target_Year,0.013006
115,cat__Institute_National Institute of Technolog...,0.010120
425,cat__Quota_HS,0.007845


In [ ]:
# ==========================================
# STEP 9: ROBUST HISTORICAL FEATURES
# ==========================================

print("====================================")
print("ROBUST FEATURE RANDOM FOREST")
print("====================================")


# --------------------------------------
# Features
# --------------------------------------

robust_numerical_features = [
    "Target_Year",
    "Historical_Count",
    "Historical_Mean",
    "Historical_Median",
    "Recent_Mean",
    "Trend_Slope"
]


robust_features = (
    categorical_features +
    robust_numerical_features
)


# --------------------------------------
# X datasets
# --------------------------------------

X_train_robust = train_data[robust_features]

X_validation_robust = validation_data[robust_features]


# --------------------------------------
# Preprocessor
# --------------------------------------

robust_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


robust_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            robust_numeric_transformer,
            robust_numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# --------------------------------------
# Random Forest
# --------------------------------------

rf_robust_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_robust_pipeline = Pipeline(
    steps=[
        ("preprocessor", robust_preprocessor),
        ("model", rf_robust_model)
    ]
)


# --------------------------------------
# Train
# --------------------------------------

print("Training robust model...")

rf_robust_pipeline.fit(
    X_train_robust,
    y_train
)

print("Model training completed!")


# --------------------------------------
# Predict 2025
# --------------------------------------

print("\nMaking 2025 predictions...")

y_pred_2025_robust = rf_robust_pipeline.predict(
    X_validation_robust
)


# --------------------------------------
# Evaluation
# --------------------------------------

mae_robust = mean_absolute_error(
    y_validation,
    y_pred_2025_robust
)

rmse_robust = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_2025_robust
    )
)

r2_robust = r2_score(
    y_validation,
    y_pred_2025_robust
)


print("\n====================================")
print("2025 ROBUST MODEL RESULTS")
print("====================================")

print("MAE :", round(mae_robust, 2))
print("RMSE:", round(rmse_robust, 2))
print("R²  :", round(r2_robust, 4))

ROBUST FEATURE RANDOM FOREST
Training robust model...
Model training completed!

Making 2025 predictions...

2025 ROBUST MODEL RESULTS
MAE : 2715.91
RMSE: 16349.9
R²  : 0.8224


In [ ]:
# ==========================================
# STEP 10: 2026 PREDICTION
# ==========================================

print("====================================")
print("PREDICTING 2026 CLOSING RANKS")
print("====================================")


# --------------------------------------
# Prepare 2026 features
# --------------------------------------

X_2026_v2 = test_2026[all_features_v2]


# --------------------------------------
# Make predictions
# --------------------------------------

print("Generating 2026 predictions...")

y_pred_2026 = rf_pipeline_v2.predict(
    X_2026_v2
)


# --------------------------------------
# Add predictions to 2026 dataset
# --------------------------------------

results_2026 = test_2026.copy()

results_2026["Predicted_Closing_Rank"] = y_pred_2026

results_2026["Prediction_Error"] = (
    results_2026["Predicted_Closing_Rank"]
    - results_2026["Target_Closing_Rank"]
)

results_2026["Absolute_Error"] = (
    abs(results_2026["Prediction_Error"])
)


print("\n2026 predictions generated!")

print("\nFirst 10 predictions:")

display(
    results_2026[
        [
            "Institute",
            "Academic Program Name",
            "Quota",
            "Seat Type",
            "Gender",
            "Target_Closing_Rank",
            "Predicted_Closing_Rank",
            "Absolute_Error"
        ]
    ].head(10)
)

PREDICTING 2026 CLOSING RANKS
Generating 2026 predictions...

2026 predictions generated!

First 10 predictions:


,Institute,Academic Program Name,Quota,Seat Type,Gender,Target_Closing_Rank,Predicted_Closing_Rank,Absolute_Error
70873,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,EWS,Gender-Neutral,11452.0,3935.819412,7516.180588
70874,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Female-only (including Supernumerary),25049.0,8976.767847,16072.232153
70875,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OBC-NCL,Gender-Neutral,25175.0,9136.898845,16038.101155
70876,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN,Gender-Neutral,70998.0,23291.436982,47706.563018
70877,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,OPEN (PwD),Gender-Neutral,3737.0,1750.384011,1986.615989
70878,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,SC,Female-only (including Supernumerary),12646.0,4919.553172,7726.446828
70879,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,SC,Gender-Neutral,11548.0,4946.475315,6601.524685
70880,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",AI,ST,Gender-Neutral,3076.0,2155.495566,920.504434
70881,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",HS,EWS,Gender-Neutral,20473.0,6563.451540,13909.548460
70882,"Assam University, Silchar","Agricultural Engineering (4 Years, Bachelor of...",HS,OBC-NCL,Female-only (including Supernumerary),75635.0,12451.624849,63183.375151


In [ ]:
# ==========================================
# STEP 11: 2026 MODEL EVALUATION
# ==========================================

print("====================================")
print("2026 MODEL EVALUATION")
print("====================================")


# --------------------------------------
# Calculate metrics
# --------------------------------------

mae_2026 = mean_absolute_error(
    y_2026,
    y_pred_2026
)

rmse_2026 = np.sqrt(
    mean_squared_error(
        y_2026,
        y_pred_2026
    )
)

r2_2026 = r2_score(
    y_2026,
    y_pred_2026
)


print("\n====================================")
print("2026 RESULTS")
print("====================================")

print("MAE :", round(mae_2026, 2))
print("RMSE:", round(rmse_2026, 2))
print("R²  :", round(r2_2026, 4))


# --------------------------------------
# Additional error information
# --------------------------------------

print("\n====================================")
print("ERROR ANALYSIS")
print("====================================")

print(
    "Median Absolute Error:",
    round(
        np.median(
            results_2026["Absolute_Error"]
        ),
        2
    )
)

print(
    "Maximum Absolute Error:",
    round(
        results_2026["Absolute_Error"].max(),
        2
    )
)


# --------------------------------------
# Prediction correlation
# --------------------------------------

correlation = np.corrcoef(
    y_2026,
    y_pred_2026
)[0, 1]

print(
    "Actual vs Predicted Correlation:",
    round(correlation, 4)
)

2026 MODEL EVALUATION

2026 RESULTS
MAE : 8508.6
RMSE: 26237.15
R²  : 0.5301

ERROR ANALYSIS
Median Absolute Error: 2916.93
Maximum Absolute Error: 1037069.96
Actual vs Predicted Correlation: 0.7286


In [ ]:
# ==========================================
# STEP 12: ANALYZE 2026 LARGE ERRORS
# ==========================================

print("====================================")
print("2026 LARGE ERROR ANALYSIS")
print("====================================")


# Sort by absolute error
largest_errors = results_2026.sort_values(
    by="Absolute_Error",
    ascending=False
)


# Show top 20
display(
    largest_errors[
        [
            "Institute",
            "Academic Program Name",
            "Quota",
            "Seat Type",
            "Gender",
            "Target_Closing_Rank",
            "Predicted_Closing_Rank",
            "Prediction_Error",
            "Absolute_Error"
        ]
    ].head(20)
)

2026 LARGE ERROR ANALYSIS


,Institute,Academic Program Name,Quota,Seat Type,Gender,Target_Closing_Rank,Predicted_Closing_Rank,Prediction_Error,Absolute_Error
81262,"National Institute of Technology, Mizoram",Electronics and Communication Engineering (4 Y...,HS,OPEN,Female-only (including Supernumerary),1497038.0,459968.038313,-1.037070e+06,1.037070e+06
81216,"National Institute of Technology, Mizoram","Chemical Science and Technology (4 Years, Bach...",HS,OPEN,Gender-Neutral,1032951.0,467375.038942,-5.655760e+05,5.655760e+05
81210,"National Institute of Technology, Manipur","Mechanical Engineering (5 Years, Bachelor and ...",HS,OPEN,Gender-Neutral,514584.0,23412.933155,-4.911711e+05,4.911711e+05
81275,"National Institute of Technology, Mizoram","Mathematics and Computing (4 Years, Bachelor o...",HS,OPEN,Female-only (including Supernumerary),936083.0,473447.129560,-4.626359e+05,4.626359e+05
81167,"National Institute of Technology, Manipur","Electrical Engineering (5 Years, Bachelor and ...",HS,OPEN,Gender-Neutral,479380.0,23412.933155,-4.559671e+05,4.559671e+05
81847,"National Institute of Technology, Srinagar","Chemical Engineering (4 Years, Bachelor of Tec...",JK,OBC-NCL,Gender-Neutral,581831.0,127625.481340,-4.542055e+05,4.542055e+05
81188,"National Institute of Technology, Manipur",Electronics and Communication Engineering (5 Y...,HS,OPEN,Gender-Neutral,469151.0,23412.933155,-4.457381e+05,4.457381e+05
81282,"National Institute of Technology, Mizoram","Mechanical Engineering (4 Years, Bachelor of T...",HS,OPEN,Gender-Neutral,881407.0,465520.126665,-4.158869e+05,4.158869e+05
81263,"National Institute of Technology, Mizoram",Electronics and Communication Engineering (4 Y...,HS,OPEN,Gender-Neutral,838800.0,467704.398605,-3.710956e+05,3.710956e+05
81235,"National Institute of Technology, Mizoram","Computer Science and Engineering (4 Years, Bac...",HS,OBC-NCL,Gender-Neutral,539090.0,196847.714262,-3.422423e+05,3.422423e+05


In [ ]:
# ==========================================
# STEP 13: LOG TARGET + TARGET YEAR
# ==========================================

print("====================================")
print("LOG TARGET + TARGET YEAR")
print("====================================")


# --------------------------------------
# Log-transform training target
# --------------------------------------

y_train_log_v2 = np.log1p(y_train)


# --------------------------------------
# Random Forest
# --------------------------------------

rf_log_year_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_log_year_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", rf_log_year_model)
    ]
)


# --------------------------------------
# Train
# --------------------------------------

print("Training model...")

rf_log_year_pipeline.fit(
    X_train_v2,
    y_train_log_v2
)

print("Model training completed!")


# --------------------------------------
# 2025 prediction
# --------------------------------------

print("\nMaking 2025 predictions...")

y_pred_2025_log_year = (
    rf_log_year_pipeline.predict(
        X_validation_v2
    )
)


# Convert back to original rank
y_pred_2025_log_year_original = np.expm1(
    y_pred_2025_log_year
)


# --------------------------------------
# 2025 evaluation
# --------------------------------------

mae_log_year = mean_absolute_error(
    y_validation,
    y_pred_2025_log_year_original
)

rmse_log_year = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_2025_log_year_original
    )
)

r2_log_year = r2_score(
    y_validation,
    y_pred_2025_log_year_original
)


print("\n====================================")
print("2025 LOG + YEAR RESULTS")
print("====================================")

print("MAE :", round(mae_log_year, 2))
print("RMSE:", round(rmse_log_year, 2))
print("R²  :", round(r2_log_year, 4))

LOG TARGET + TARGET YEAR
Training model...
Model training completed!

Making 2025 predictions...

2025 LOG + YEAR RESULTS
MAE : 2626.25
RMSE: 15907.5
R²  : 0.8319


In [ ]:
# ==========================================
# STEP 14: 2026 TEST - LOG + YEAR MODEL
# ==========================================

print("====================================")
print("2026 TEST - LOG + YEAR MODEL")
print("====================================")


# --------------------------------------
# Predict 2026
# --------------------------------------

print("Generating 2026 predictions...")

y_pred_2026_log_year = (
    rf_log_year_pipeline.predict(
        X_2026_v2
    )
)


# Convert log prediction back to rank
y_pred_2026_log_year_original = np.expm1(
    y_pred_2026_log_year
)


# --------------------------------------
# Evaluation
# --------------------------------------

mae_2026_log_year = mean_absolute_error(
    y_2026,
    y_pred_2026_log_year_original
)

rmse_2026_log_year = np.sqrt(
    mean_squared_error(
        y_2026,
        y_pred_2026_log_year_original
    )
)

r2_2026_log_year = r2_score(
    y_2026,
    y_pred_2026_log_year_original
)


# --------------------------------------
# Median error
# --------------------------------------

median_error_2026_log_year = np.median(
    np.abs(
        y_2026 -
        y_pred_2026_log_year_original
    )
)


# --------------------------------------
# Correlation
# --------------------------------------

correlation_2026_log_year = np.corrcoef(
    y_2026,
    y_pred_2026_log_year_original
)[0, 1]


print("\n====================================")
print("2026 LOG + YEAR RESULTS")
print("====================================")

print("MAE :", round(mae_2026_log_year, 2))
print("RMSE:", round(rmse_2026_log_year, 2))
print("R²  :", round(r2_2026_log_year, 4))

print(
    "Median Absolute Error:",
    round(median_error_2026_log_year, 2)
)

print(
    "Actual vs Predicted Correlation:",
    round(correlation_2026_log_year, 4)
)

2026 TEST - LOG + YEAR MODEL
Generating 2026 predictions...

2026 LOG + YEAR RESULTS
MAE : 7837.7
RMSE: 27030.97
R²  : 0.5013
Median Absolute Error: 2427.37
Actual vs Predicted Correlation: 0.7615


In [ ]:
# ==========================================
# STEP 15: ERROR BY RANK RANGE
# ==========================================

print("====================================")
print("2026 ERROR BY RANK RANGE")
print("====================================")


analysis_2026 = pd.DataFrame({
    "Actual": y_2026.values,
    "Predicted": y_pred_2026_log_year_original
})


# --------------------------------------
# Create rank groups
# --------------------------------------

def rank_group(rank):

    if rank <= 5000:
        return "1 - 5,000"

    elif rank <= 10000:
        return "5,001 - 10,000"

    elif rank <= 25000:
        return "10,001 - 25,000"

    elif rank <= 50000:
        return "25,001 - 50,000"

    elif rank <= 100000:
        return "50,001 - 1,00,000"

    elif rank <= 500000:
        return "1,00,001 - 5,00,000"

    else:
        return "500,000+"


analysis_2026["Rank_Group"] = (
    analysis_2026["Actual"]
    .apply(rank_group)
)


analysis_2026["Absolute_Error"] = (
    abs(
        analysis_2026["Actual"]
        -
        analysis_2026["Predicted"]
    )
)


# --------------------------------------
# Group analysis
# --------------------------------------

group_results = (
    analysis_2026
    .groupby("Rank_Group")
    .agg(
        Records=("Actual", "count"),
        Average_Actual=("Actual", "mean"),
        Average_Predicted=("Predicted", "mean"),
        MAE=("Absolute_Error", "mean")
    )
    .reset_index()
)


# --------------------------------------
# Order groups
# --------------------------------------

group_order = [
    "1 - 5,000",
    "5,001 - 10,000",
    "10,001 - 25,000",
    "25,001 - 50,000",
    "50,001 - 1,00,000",
    "1,00,001 - 5,00,000",
    "500,000+"
]


group_results["Rank_Group"] = pd.Categorical(
    group_results["Rank_Group"],
    categories=group_order,
    ordered=True
)


group_results = group_results.sort_values(
    "Rank_Group"
)


display(group_results)

2026 ERROR BY RANK RANGE


,Rank_Group,Records,Average_Actual,Average_Predicted,MAE
0,"1 - 5,000",6672,1895.874550,3268.501891,1833.822854
4,"5,001 - 10,000",2338,7162.883661,7166.758253,3951.570025
2,"10,001 - 25,000",2297,15606.127558,13531.942412,8276.051483
3,"25,001 - 50,000",1066,35449.121013,23850.509983,14211.350044
5,"50,001 - 1,00,000",514,66253.834630,33767.360082,35024.029890
1,"1,00,001 - 5,00,000",185,193561.827027,81209.429156,119616.359662
6,"500,000+",13,777785.615385,255358.583573,522427.031811


In [ ]:
# ==========================================
# STEP 16: LOG + YEAR + 300 TREES
# ==========================================

print("====================================")
print("RANDOM FOREST - 300 TREES")
print("====================================")


rf_300_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_300_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", rf_300_model)
    ]
)


# --------------------------------------
# Train
# --------------------------------------

print("Training 300-tree model...")

rf_300_pipeline.fit(
    X_train_v2,
    y_train_log_v2
)

print("Model training completed!")


# --------------------------------------
# 2025 validation
# --------------------------------------

print("\nEvaluating 2025...")

pred_2025_300_log = rf_300_pipeline.predict(
    X_validation_v2
)

pred_2025_300 = np.expm1(
    pred_2025_300_log
)


mae_300_2025 = mean_absolute_error(
    y_validation,
    pred_2025_300
)

rmse_300_2025 = np.sqrt(
    mean_squared_error(
        y_validation,
        pred_2025_300
    )
)

r2_300_2025 = r2_score(
    y_validation,
    pred_2025_300
)


print("\n2025 RESULTS")
print("MAE :", round(mae_300_2025, 2))
print("RMSE:", round(rmse_300_2025, 2))
print("R²  :", round(r2_300_2025, 4))


# --------------------------------------
# 2026 test
# --------------------------------------

print("\nEvaluating 2026...")

pred_2026_300_log = rf_300_pipeline.predict(
    X_2026_v2
)

pred_2026_300 = np.expm1(
    pred_2026_300_log
)


mae_300_2026 = mean_absolute_error(
    y_2026,
    pred_2026_300
)

rmse_300_2026 = np.sqrt(
    mean_squared_error(
        y_2026,
        pred_2026_300
    )
)

r2_300_2026 = r2_score(
    y_2026,
    pred_2026_300
)

median_300_2026 = np.median(
    np.abs(
        y_2026 - pred_2026_300
    )
)


print("\n2026 RESULTS")
print("MAE :", round(mae_300_2026, 2))
print("RMSE:", round(rmse_300_2026, 2))
print("R²  :", round(r2_300_2026, 4))
print(
    "Median Absolute Error:",
    round(median_300_2026, 2)
)

RANDOM FOREST - 300 TREES
Training 300-tree model...
Model training completed!

Evaluating 2025...

2025 RESULTS
MAE : 2634.42
RMSE: 15995.66
R²  : 0.83

Evaluating 2026...

2026 RESULTS
MAE : 7786.83
RMSE: 26709.65
R²  : 0.513
Median Absolute Error: 2437.07


In [ ]:
print("df:", "df" in globals())
print("rf_log_year_pipeline:", "rf_log_year_pipeline" in globals())
print("X_train_v2:", "X_train_v2" in globals())
print("y_train_log_v2:", "y_train_log_v2" in globals())
print("pred_2026_300:", "pred_2026_300" in globals())

df: True
rf_log_year_pipeline: True
X_train_v2: True
y_train_log_v2: True
pred_2026_300: True


In [28]:
import numpy as np
import pandas as pd

# Analyze error distribution across actual 2026 rank tiers
test_analysis = pd.DataFrame({
    "Actual": y_2026.values,
    "Predicted": pred_2026_300
})

def categorize_rank(rank):
    if rank <= 5000: return "1 - 5,000"
    elif rank <= 10000: return "5,001 - 10,000"
    elif rank <= 25000: return "10,001 - 25,000"
    elif rank <= 50000: return "25,001 - 50,000"
    elif rank <= 100000: return "50,001 - 100,000"
    else: return "100,000+"

test_analysis["Rank_Group"] = test_analysis["Actual"].apply(categorize_rank)
test_analysis["Absolute_Error"] = np.abs(test_analysis["Actual"] - test_analysis["Predicted"])

rank_summary = test_analysis.groupby("Rank_Group").agg(
    Count=("Actual", "count"),
    MAE=("Absolute_Error", "mean"),
    Median_AE=("Absolute_Error", "median")
).reindex(["1 - 5,000", "5,001 - 10,000", "10,001 - 25,000", "25,001 - 50,000", "50,001 - 100,000", "100,000+"])
print(rank_summary)

                  Count            MAE      Median_AE
Rank_Group                                           
1 - 5,000          6672    1827.248829    1010.222482
5,001 - 10,000     2338    3917.390411    2940.884086
10,001 - 25,000    2297    8144.863822    7304.980161
25,001 - 50,000    1066   14221.051727   14086.767996
50,001 - 100,000    514   35077.017588   33921.162068
100,000+            198  144658.833361  100356.327040


In [30]:
import joblib
import numpy as np

import pandas as pd

# ---------------------------------------------------------
# 1. RETRAIN ON FULL HISTORICAL DATA (2019 - 2025)
# ---------------------------------------------------------
X_full_train = pd.concat([X_train_v2, X_validation_v2], axis=0)
y_full_train_log = np.concat([y_train_log_v2, np.log1p(y_validation)])

# Fit 300-tree model pipeline on entire dataset
rf_300_pipeline.fit(X_full_train, y_full_train_log)

# ---------------------------------------------------------
# 2. GENERATE FINAL 2026 PREDICTIONS
# ---------------------------------------------------------
preds_2026_log = rf_300_pipeline.predict(X_2026_v2)

# Create exported DataFrame
ml_results_2026 = test_2026.copy()
ml_results_2026["Predicted_2026_Closing_Rank"] = np.expm1(preds_2026_log)

# Calculate prediction deviation bounds (Uncertainty Range)
ml_results_2026["Predicted_Min_Rank"] = np.maximum(
    1, ml_results_2026["Predicted_2026_Closing_Rank"] * 0.90
)
ml_results_2026["Predicted_Max_Rank"] = (
    ml_results_2026["Predicted_2026_Closing_Rank"] * 1.10
)

# ---------------------------------------------------------
# 3. SAVE ARTIFACTS FOR STREAMLIT
# ---------------------------------------------------------
# Export predictions table
ml_results_2026.to_csv("josaa_2026_ml_predictions.csv", index=False)

# Export full trained model pipeline
joblib.dump(rf_300_pipeline, "josaa_rf_model.pkl")

print("Files successfully generated:")
print("1. 'josaa_2026_ml_predictions.csv'")
print("2. 'josaa_rf_model.pkl'")

Files successfully generated:
1. 'josaa_2026_ml_predictions.csv'
2. 'josaa_rf_model.pkl'


In [34]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.2 MB/s eta 0:00:00


In [35]:
%%writefile app.py
import joblib
import numpy as np
import pandas as pd
import streamlit as st

# Page Configuration
st.set_page_config(
    page_title="JoSAA ML College Predictor 2026",
    page_icon="🎓",
    layout="wide",
)

# Load Assets
@st.cache_resource
def load_assets():
    model = joblib.load("josaa_rf_model.pkl")
    predictions_df = pd.read_csv("josaa_2026_ml_predictions.csv")
    return model, predictions_df

try:
    model, df = load_assets()
except Exception as e:
    st.error("Please run Step 19 first to generate 'josaa_rf_model.pkl' and 'josaa_2026_ml_predictions.csv'.")
    st.stop()

# Title Header
st.title("🎓 JoSAA College & Branch Recommendation System")
st.markdown("Predict expected closing ranks and explore **Safe, Target, and Dream** options powered by Random Forest ML.")

# Sidebar User Inputs
st.sidebar.header("🔍 Candidate Inputs")

user_rank = st.sidebar.number_input(
    "Enter Your JEE Main / Advanced Rank:",
    min_value=1,
    max_value=1000000,
    value=12000,
    step=500,
)

seat_type = st.sidebar.selectbox(
    "Category (Seat Type):", sorted(df["Seat Type"].dropna().unique())
)

gender = st.sidebar.selectbox(
    "Gender:", sorted(df["Gender"].dropna().unique())
)

quota = st.sidebar.selectbox("Quota:", sorted(df["Quota"].dropna().unique()))

# Optional Branch Filter
all_branches = sorted(df["Academic Program Name"].dropna().unique())
selected_branches = st.sidebar.multiselect(
    "Filter Branch Preferences (Optional):", all_branches
)

# Optional Institute Filter
all_institutes = sorted(df["Institute"].dropna().unique())
selected_institutes = st.sidebar.multiselect(
    "Filter Institute Preferences (Optional):", all_institutes
)

# Categorization Logic
def classify_admission_chance(predicted_closing, candidate_rank):
    if predicted_closing >= candidate_rank * 1.15:
        return "🟢 Safe"
    elif predicted_closing >= candidate_rank * 0.85:
        return "🟡 Target"
    else:
        return "🔴 Dream"

# Main Analysis Output
if st.sidebar.button("Get Recommendations 🚀", type="primary"):

    filtered_df = df[
        (df["Seat Type"] == seat_type)
        & (df["Gender"] == gender)
        & (df["Quota"] == quota)
    ].copy()

    if selected_branches:
        filtered_df = filtered_df[
            filtered_df["Academic Program Name"].isin(selected_branches)
        ]

    if selected_institutes:
        filtered_df = filtered_df[
            filtered_df["Institute"].isin(selected_institutes)
        ]

    if filtered_df.empty:
        st.warning("No matching college/branch records found for this combination. Try broadening your criteria.")
    else:
        filtered_df["Admission Category"] = filtered_df["Predicted_2026_Closing_Rank"].apply(
            lambda pred: classify_admission_chance(pred, user_rank)
        )

        display_df = filtered_df[
            [
                "Admission Category",
                "Institute",
                "Academic Program Name",
                "Predicted_2026_Closing_Rank",
                "Predicted_Min_Rank",
                "Predicted_Max_Rank",
                "Recent_Mean",
            ]
        ].copy()

        display_df.rename(
            columns={
                "Admission Category": "Category",
                "Predicted_2026_Closing_Rank": "Predicted Closing Rank",
                "Predicted_Min_Rank": "Lower Bound (90%)",
                "Predicted_Max_Rank": "Upper Bound (110%)",
                "Recent_Mean": "Historical Avg (Recent)",
            },
            inplace=True,
        )

        display_df.sort_values(by="Predicted Closing Rank", ascending=True, inplace=True)

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Options", len(display_df))
        c2.metric("🟢 Safe", len(display_df[display_df["Category"] == "🟢 Safe"]))
        c3.metric("🟡 Target", len(display_df[display_df["Category"] == "🟡 Target"]))
        c4.metric("🔴 Dream", len(display_df[display_df["Category"] == "🔴 Dream"]))

        st.markdown("---")
        st.subheader("📋 Recommended Options")

        st.dataframe(
            display_df.style.format(
                {
                    "Predicted Closing Rank": "{:.0f}",
                    "Lower Bound (90%)": "{:.0f}",
                    "Upper Bound (110%)": "{:.0f}",
                    "Historical Avg (Recent)": "{:.0f}",
                }
            ),
            use_container_width=True,
            height=500,
        )

Writing app.py
